# 04. Modelo dimensional y carga Gold

Consulta `data/oro/bookeau.sqlite3`, construida por `src/construir_gold_bookeau.py` (paso `gold`), y los controles de conciliación en `docs/artifacts/transformacion/`. El DDL está en `sql/00_crear_modelo.sql` y las consultas de los análisis en `sql/01_consultas_analisis.sql`.

`dim_estado` guarda el estado original, el analítico (R01, R06, R07) y el grupo de estado (R04).

**Antes de ejecutar:** desde la raíz del proyecto, correr `python3 src/main.py --dummy` (datos ficticios versionados) o `python3 src/main.py` (datos reales, que no se versionan). El cuaderno solo lee los resultados del pipeline en `docs/artifacts/`, `data/plata/` y `data/oro/`. Con datos ficticios los conteos son pequeños y no coinciden con las cifras del informe, que usan los datos reales.


In [1]:
from pathlib import Path
import sqlite3
import json
from IPython.display import display, Markdown

PROYECTO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "main.py").is_file()), None)
if PROYECTO is None:
    raise RuntimeError("Abre el cuaderno desde la carpeta del proyecto o desde notebooks/.")
if not (PROYECTO / "data/oro/bookeau.sqlite3").is_file():
    raise RuntimeError("Falta Gold. Ejecuta primero desde la raíz: python3 src/main.py --dummy")
db = sqlite3.connect(f"file:{PROYECTO / 'data/oro/bookeau.sqlite3'}?mode=ro", uri=True)
db.row_factory = sqlite3.Row

def consulta(sql):
    filas = [dict(r) for r in db.execute(sql)]
    if not filas:
        print("Sin filas")
        return
    columnas = list(filas[0])
    esc = lambda x: str(x).replace("|", "\|").replace("\n", " ")
    tabla = ["| " + " | ".join(columnas) + " |", "| " + " | ".join(["---"]*len(columnas)) + " |"]
    tabla.extend("| " + " | ".join(esc(fila[c]) for c in columnas) + " |" for fila in filas)
    display(Markdown("\n".join(tabla)))


<>:20: SyntaxWarning: invalid escape sequence '\|'
<>:20: SyntaxWarning: invalid escape sequence '\|'
/var/folders/jq/9m84jg7x2hz0l32cqnwklfbm0000gn/T/ipykernel_48270/673936807.py:20: SyntaxWarning: invalid escape sequence '\|'
  esc = lambda x: str(x).replace("|", "\|").replace("\n", " ")


## 1. Controles de la carga


In [2]:
controles = json.loads((PROYECTO / "docs" / "artifacts" / "transformacion" / "controles_gold.json").read_text())
print(json.dumps(controles, ensure_ascii=False, indent=2))
consulta("SELECT estado_original, estado_analitico, grupo_estado FROM dim_estado")


{
  "conteos": {
    "dim_fecha": 1753,
    "dim_hora": 1440,
    "dim_periodo": 9,
    "dim_dia_semana": 7,
    "dim_servicio": 5,
    "dim_modalidad": 9,
    "dim_estado": 12,
    "dim_programa": 8,
    "dim_tipo_encuesta": 4,
    "dim_pregunta": 22,
    "hecho_reserva": 1709,
    "hecho_encuesta": 848,
    "hecho_respuesta": 6464,
    "hecho_oferta": 435
  },
  "reservas_silver": 1972,
  "reservas_excluidas_R03": 263,
  "reservas_gold": 1709,
  "franjas_oferta_silver": 435,
  "franjas_oferta_gold": 435,
  "encuestas_excluidas_R03": 70,
  "atenciones": 862,
  "encuestas_excluidas_R02": 9,
  "claves_foraneas": "OK",
  "unicidad_granos": "OK",
  "conciliacion": "OK"
}


| estado_original | estado_analitico | grupo_estado |
| --- | --- | --- |
| Cancelada | Cancelada | Cancelada |
| Cancelada a tiempo | Cancelada | Cancelada |
| Cancelada con excusa | Cancelada | Cancelada |
| Cancelada por sanción | Cancelada | Cancelada |
| Cancelada y sancionada | Cancelada | Cancelada |
| Cola cancelada por reservación | Cola cancelada por reservación | Cola a reserva |
| En cola | En cola | Cola |
| En ejecución | Atendida | Atendida |
| Finalizada | Atendida | Atendida |
| No asistió | No asistió | No asistió |
| Realizada | Atendida | Atendida |
| Reservada | No asistió | No asistió |

## 2. Reserva y atención registrada

No hay oferta general disponible; los eventos no miden ocupación. Las medidas se agregan directamente desde hecho_reserva.


In [3]:
consulta("""
SELECT p.periodo_original, COUNT(*) eventos,
       SUM(CASE WHEN e.estado_analitico='Atendida' THEN 1 ELSE 0 END) atenciones
FROM hecho_reserva h JOIN dim_periodo p USING(sk_periodo)
JOIN dim_estado e USING(sk_estado)
GROUP BY p.periodo_original ORDER BY p.periodo_original
""")


| periodo_original | eventos | atenciones |
| --- | --- | --- |
| 201710 | 220 | 93 |
| 201719 | 184 | 83 |
| 201720 | 209 | 107 |
| 201819 | 208 | 113 |
| 201820 | 185 | 114 |
| 201920 | 182 | 92 |
| 202010 | 199 | 93 |
| 202110 | 167 | 85 |
| 202120 | 155 | 82 |

## 3. Satisfacción

Solo encuestas posteriores incluidas; el denominador de completitud es encuesta incluida, no población invitada. La media es descriptiva y debe acompañarse de n y distribución.


In [4]:
consulta("""
SELECT m.tipo_horario_original, COUNT(*) encuestas,
       COUNT(h.calificacion_ayuda_tutor) n_calificaciones,
       ROUND(AVG(h.calificacion_ayuda_tutor),4) media_descriptiva
FROM hecho_encuesta h JOIN dim_tipo_encuesta t USING(sk_tipo_encuesta)
JOIN dim_modalidad m USING(sk_modalidad)
WHERE t.etapa='posterior' GROUP BY m.tipo_horario_original
""")


| tipo_horario_original | encuestas | n_calificaciones | media_descriptiva |
| --- | --- | --- | --- |
| Express | 148 | 92 | 4.5761 |
| Normal | 331 | 205 | 4.4195 |
| Normal pico | 48 | 28 | 4.25 |

## 4. Reconstruir Gold y tablero

La reconstrucción usa Silver. Cerrar la conexión antes de reemplazar la base. El tablero local se genera sin publicar nada fuera del equipo.


In [5]:
# db.close()
# import runpy, sys
# sys.path.insert(0, str(PROYECTO / "src"))
# runpy.run_path(str(PROYECTO / "src" / "construir_gold_bookeau.py"), run_name="__main__")
# runpy.run_path(str(PROYECTO / "src" / "generar_tablero_bookeau.py"), run_name="__main__")
